# IMDB Sentiment Analysis
Train a binary classifier (positive=1 / negative=0) on IMDB movie reviews, then upload to the HTB evaluation portal.

In [ ]:
# ── 0. Install dependencies if needed ────────────────────────────────────────
# !pip install scikit-learn pandas requests joblib nltk

In [ ]:
import io, json, os, sys, zipfile, tarfile
import joblib, requests
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline

MODEL_FILE = "skills_assessment.joblib"
DATA_DIR   = "imdb_data"
HTB_URL    = "https://academy.hackthebox.com/storage/modules/290/imdb_sentiment_dataset.zip"

## 1. Download dataset
Paste the dataset URL from the HTB question page. If the download fails the cell falls back to NLTK's `movie_reviews` corpus.

In [ ]:
# ← Paste the URL from the HTB challenge page here
DATASET_URL = "https://academy.hackthebox.com/storage/modules/290/imdb_sentiment_dataset.zip"
DATA_DIR = "imdb_data"
MODEL_FILE = "imdb_sentiment_model.joblib"

os.makedirs(DATA_DIR, exist_ok=True)

def download(url):
    r = requests.get(url, timeout=60)
    if r.status_code != 200:
        print(f"Download failed (HTTP {r.status_code})")
        return False
    if url.endswith(".tar.gz") or url.endswith(".tgz"):
        with tarfile.open(fileobj=io.BytesIO(r.content)) as t:
            t.extractall(DATA_DIR)
    else:
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            z.extractall(DATA_DIR)
            print("Contents:", z.namelist())
    print("Download + extraction OK")
    return True

download(DATASET_URL)

In [ ]:
train_p = os.path.join(DATA_DIR, "train.json")
test_p  = os.path.join(DATA_DIR, "test.json")

with open(train_p, encoding="utf-8") as f:
    train_data = json.load(f)
with open(test_p, encoding="utf-8") as f:
    test_data = json.load(f)

X_train = [s["text"] for s in train_data]
y_train = [s["label"] for s in train_data]
X_test  = [s["text"] for s in test_data]
y_test  = [s["label"] for s in test_data]

print(f"Train: {len(X_train)}  Test: {len(X_test)}")
print(f"Sample: {X_train[0][:80]!r}  label={y_train[0]}")

## 2. Preprocess

In [ ]:
pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer()),
    ("classifier", LogisticRegression(max_iter=1000)),
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

## 3. Train

In [ ]:
# sklearn <=1.6 TfidfTransformer.transform() requires _idf_diag (sparse CSR
# diagonal IDF matrix). sklearn >=1.8 removed it in favour of idf_ directly.
# Inject _idf_diag before saving so the server's older sklearn can predict().
tfidf_trans = pipeline.named_steps["vectorizer"]._tfidf
if not hasattr(tfidf_trans, "_idf_diag"):
    idf = tfidf_trans.idf_
    n   = len(idf)
    tfidf_trans._idf_diag = sp.diags(idf, 0, shape=(n, n), format="csr", dtype=idf.dtype)

# protocol=4: readable by Python 3.4+ (our Python 3.14 defaults to protocol 5)
joblib.dump(pipeline, MODEL_FILE, compress=False, protocol=4)
print(f"Saved → {MODEL_FILE}  ({os.path.getsize(MODEL_FILE)/1e3:.0f} KB)")

## 4. Save model

In [ ]:
TARGET_IP = "<playground-vm-ip>"   # ← replace with your VM IP

url = f"http://{TARGET_IP}:5000/api/upload"
with open(MODEL_FILE, "rb") as f:
    response = requests.post(url, files={"model": f}, timeout=60)

print(json.dumps(response.json(), indent=4))

## 5. Upload to HTB evaluation portal
Set `TARGET_IP` to the Playground VM IP shown on the HTB challenge page.

In [ ]:
TARGET_IP = "<playground-vm-ip>"   # ← replace with your VM IP

url = f"http://{TARGET_IP}:5000/api/upload"
with open(MODEL_FILE, "rb") as f:
    response = requests.post(url, files={"model": f}, timeout=60)

print(f"HTTP {response.status_code}")
try:
    print(json.dumps(response.json(), indent=4))
except Exception:
    print(response.text)